In [66]:
from pgljupyter import *
from openalea.lpy import *
from openalea.plantgl.all import *

In [ ]:
help('modules')

In [67]:
l = Lsystem('../lpy/leuwenberg.lpy')
tree = l.axiom
step = 0
trees = [tree]
scenes = [l.sceneInterpretation(tree)]
while step < l.derivationLength:
    trees.append(l.derive(trees[-1], len(trees), 1))
    scenes.append(l.sceneInterpretation(trees[-1]))
    step = step + 1

In [73]:
def to_json(geom, ms):
    if isinstance(geom, Sphere):
        return [0, [geom.radius, geom.slices, geom.stacks], ms]
    elif isinstance(geom, Cylinder):
        return [1, [geom.radius, geom.height, geom.slices, geom.solid], ms]
    elif isinstance(geom, Box):
        return [2, list(geom.size)]
    elif isinstance(geom, Cone):
        return [3, [geom.radius, geom.height, geom.slices, geom.solid], ms]
    elif isinstance(geom, TriangleSet):
        return [4, [
            [p for point in geom.pointList for p in point],
            [i for index in geom.indexList for i in index]
        ], ms]
    elif isinstance(geom, Extrusion):
        if isinstance(geom.axis, Polyline):
            return [5, [0,
                [p for point in geom.crossSection for p in point],
                [
                    [p for point in geom.axis for p in point],
                    [p for point in geom.scaleList for p in point]
                ],
                geom.axis.stride,
                geom.axis.getLength()
            ], ms]
        elif isinstance(geom.axis, NurbsCurve):
            return [5, [1,
                [p for point in geom.crossSection for p in point],
                [
                    geom.axis.degree,
                    geom.axis.knots,
                    [p for point in geom.axis for p in point],
                    geom.startKnot,
                    geom.endKnot
                ]
            ], ms]
        else:
            print(geom.axis)
            return None
    else:
        print(geom)
        return None

def serialize(scene):
    instances = {}
    serialized = []                
    for shape in scene:
        geom = shape.geometry
        m = Matrix4()

        while (hasattr(geom, 'geometry')):
            m = m * geom.transformation().getMatrix()
            geom = geom.geometry

        if geom.getObjectId() not in instances:
            instances[geom.getObjectId()] = (geom, m, [m.data()])
        else:
            root = instances[geom.getObjectId()]
            root[2].append(m.data())

    for id in instances:
        geom, m, ms = instances[id]
        geom_ = to_json(geom, ms)
        if geom_ is not None:
            serialized.append(geom_)
    return serialized

In [74]:
serialized = serialize(scenes[-1])

In [75]:
SceneWidget(serialized)

SceneWidget(scene=[[4, [[0.0, 0.0, 0.5, 0.0, 0.0, 0.0, 0.0, -0.25, 0.3333333333333333, 0.0, -0.25, 0.666666666…